In [0]:
from pyspark.sql.functions import *
df_bronze= spark.read.table("workspace.projeto_energia.bronze_geracao_distribuida")

In [0]:
# 2. Aplicar a limpeza e as regras de negócio
df_silver = (df_bronze
    # Remover os traços do CEP
    .withColumn("cep_limpo", regexp_replace(col("CodCEP"), "-", ""))
    
    # Converter a potência para número decimal (Float), trocando vírgula por ponto
    .withColumn("potencia_kw", regexp_replace(col("MdaPotenciaInstaladaKW"), ",", ".").cast("float"))
    
    # Converter a quantidade de unidades de crédito para número inteiro
    .withColumn("qtd_uc_credito", col("QtdUCRecebeCredito").cast("int"))
    
    # Converter a data
    .withColumn("data_atualizacao", to_date(col("DthAtualizaCadastralEmpreend"))) 
    
    # Selecionar e renomear as colunas, incluindo agora a FONTE e a QTD DE CRÉDITO
    .select(
        col("NomRegiao").alias("regiao"),
        col("SigUF").alias("estado"),
        col("NomMunicipio").alias("cidade"),
        col("cep_limpo").alias("cep"),
        col("DscClasseConsumo").alias("classe_consumo"),
        col("DscFonteGeracao").alias("fonte_geracao"), # Ex: Radiacao Solar
        col("qtd_uc_credito"),
        col("potencia_kw"),
        col("data_atualizacao")
    )
    # Filtro de qualidade: ignorar registos sem CEP ou com potência inválida
    .filter(col("cep").isNotNull() & (col("potencia_kw") > 0))
)

# 3. Guardar a camada Silver
print("A guardar a Camada Silver...")
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_energia.silver_geracao_distribuida"))

#display(df_silver.limit(10))